# Notebook Overview
## Phase 3: Micro-Macro Merge

In this notebook, mirco and macro data will be merged to create a final dataset to use for analysis and visualisation. 

The notebook works as follows: 

- Phase 1: Merging of macro data by region 
- Phase 2: Merging of macro data by industry 

## Phase 1: Merging of macro data by region
## Step 1: Load Data

In [6]:
import pandas as pd

# Load the new micro data
micro = pd.read_csv("../data/processed/jobs_micro_cleaned_final.csv")

# Quick overview
print("Shape:", micro.shape)
print("\nColumns:", list(micro.columns))
print("\nSample of location/date/canton cols:")
print(micro[["posted_date", "canton", "location_raw", "city_clean"]].head(10))
print("\nMissing cantons:", micro["canton"].isna().sum())

Shape: (743, 20)

Columns: ['job_id', 'title', 'company', 'role', 'location_raw', 'city', 'canton', 'region', 'posted_date', 'quarter', 'macro_quarter', 'seniority', 'workload_min', 'workload_max', 'salary_available', 'salary_min', 'salary_max', 'skills', 'skill_count', 'description']

Sample of location/date/canton cols:


KeyError: "['city_clean'] not in index"

## Step 2: Inspect 

In [7]:
# Inspect 

print("Quarter samples:", micro["quarter"].value_counts())
print("\nMacro quarter samples:", micro["macro_quarter"].value_counts())
print("\nRegion samples:\n", micro["region"].value_counts())
print("\nMissing regions:", micro["region"].isna().sum())
print("\nMissing macro_quarter:", micro["macro_quarter"].isna().sum())

Quarter samples: quarter
2026Q1    695
2025Q4     29
2025Q3     11
2025Q2      3
NaT         3
2025Q1      1
2024Q4      1
Name: count, dtype: int64

Macro quarter samples: macro_quarter
2026Q1    695
2025Q4     29
2025Q3     11
2025Q2      3
NaT         3
2025Q1      1
2024Q4      1
Name: count, dtype: int64

Region samples:
 region
Zurich                      242
Lake Geneva Region          184
Espace Mittelland            88
Northwestern Switzerland     56
Eastern Switzerland          39
Central Switzerland          31
Ticino                        4
Name: count, dtype: int64

Missing regions: 99

Missing macro_quarter: 0


## Step 3: Use 2025Q4 BFS data as proxy for 2026Q1 

In [8]:
# Step 2: Create merge key, mapping 2026Q1 → 2025Q4 as proxy
micro["merge_quarter"] = micro["macro_quarter"].replace("2026Q1", "2025Q4")

# Check the result
print(micro["merge_quarter"].value_counts())
print("\nMissing merge_quarter:", micro["merge_quarter"].isna().sum())

merge_quarter
2025Q4    724
2025Q3     11
2025Q2      3
NaT         3
2025Q1      1
2024Q4      1
Name: count, dtype: int64

Missing merge_quarter: 0


## Step 4: Merge Data

In [10]:
import pandas as pd

# Load macro region data
macro_region = pd.read_csv("../notebooks/data/external/bfs_vacancies_major_region_clean.csv")

# Merge
merged = micro.merge(
    macro_region[["region", "quarter", "vacancies"]].rename(columns={
        "quarter": "merge_quarter",
        "vacancies": "vacancies_region"
    }),
    on=["region", "merge_quarter"],
    how="left"
)

# Check results
print("Shape before:", micro.shape)
print("Shape after:", merged.shape)
print("\nMatched rows (have vacancy data):", merged["vacancies_region"].notna().sum())
print("Unmatched rows (NaN vacancy):", merged["vacancies_region"].isna().sum())
print("\nSample:")
print(merged[["city", "region", "merge_quarter", "vacancies_region"]].head(10))

Shape before: (743, 21)
Shape after: (743, 22)

Matched rows (have vacancy data): 641
Unmatched rows (NaN vacancy): 102

Sample:
                     city              region merge_quarter  vacancies_region
0                Montreux  Lake Geneva Region        2025Q4           13400.0
1                Lausanne  Lake Geneva Region        2025Q4           13400.0
2         Plan-les-Ouates  Lake Geneva Region        2025Q4           13400.0
3                Givisiez   Espace Mittelland        2025Q4           14724.0
4                Lausanne  Lake Geneva Region        2025Q4           13400.0
5                  Genève  Lake Geneva Region        2025Q4           13400.0
6                   Gland  Lake Geneva Region        2025Q4           13400.0
7  Bussigny-près-Lausanne  Lake Geneva Region        2025Q4           13400.0
8               Epalinges  Lake Geneva Region        2025Q4           13400.0
9                  Genève  Lake Geneva Region        2025Q4           13400.0


## Step 5: Pre-save File with Region Merge 

In [11]:
merged.to_csv("../data/processed/jobs_merged_final.csv", index=False)
print("Saved! Final columns:", list(merged.columns))

Saved! Final columns: ['job_id', 'title', 'company', 'role', 'location_raw', 'city', 'canton', 'region', 'posted_date', 'quarter', 'macro_quarter', 'seniority', 'workload_min', 'workload_max', 'salary_available', 'salary_min', 'salary_max', 'skills', 'skill_count', 'description', 'merge_quarter', 'vacancies_region']


## Phase 2: Merging of macro data by industry 
## Step 1: Load Data

In [12]:
macro_industry = pd.read_csv("../notebooks/data/external/bfs_vacancies_economic_division_clean.csv")

print("Shape:", macro_industry.shape)
print("\nColumns:", list(macro_industry.columns))
print("\nIndustries:\n", macro_industry["industry"].unique())
print("\nQuarters:", sorted(macro_industry["quarter"].unique()))

# And check what we have on the micro side to match on
print("\nMicro - unique roles/search terms:")
print(micro["role"].value_counts())

Shape: (136, 4)

Columns: ['industry', 'quarter', 'vacancies', 'year']

Industries:
 ['10-33 Manufacturing' '24-25 Manufacture of metal products'
 '26 Manufacture of computer, electronic and optical products, watches and clocks'
 '28 Manufacture of machinery and equipment n.e.c.' '41-43 Construction'
 '45-47 Trade and repair of motor vehicles and motorcycles'
 '49-53 Transport and storage' '55-56 Hospitality' '58-63 ICT'
 '62-63 IT services' '64-66 Finance and Insurance'
 '68-75 Real estate & professional services'
 '77+79-82 Administrative & support services' '84 Public administration'
 '85 Education' '86-88 Health and social work'
 '90-96 Arts and entertainment']

Quarters: ['2024Q1', '2024Q2', '2024Q3', '2024Q4', '2025Q1', '2025Q2', '2025Q3', '2025Q4']

Micro - unique roles/search terms:
role
data engineer                448
data scientist               122
data analyst                  95
machine learning engineer     42
ai engineer                   36
Name: count, dtype: int64


## Step 2: check job descriptions and company info to merge industry codes

In [13]:
# Check the description column

print("Missing descriptions:", merged["description"].isna().sum())
print("\nSample description (first 500 chars):")
print(merged["description"].iloc[0][:500])

Missing descriptions: 0

Sample description (first 500 chars):
About the job Tu es actuellement étudiant.e ? Es-tu disponible rapidement pour une mission de 6 mois à 40% ? Maîtrises-tu Power BI ainsi que le tri et le nettoyage de données ? Es-tu reconnu.e pour ton esprit structuré et ta capacité de synthèse ? Mission étudiante 40% : Data Analyst Localisation: Riviera Date de début: Dès que possible Durée: Temps partiel, 40%- 6 mois de mission Type de travail: Mission A propos du rôle Rejoignez notre client, une compagnie ferroviaire en pleine transformation


In [14]:
# Check the company column

print("Missing companies:", merged["company"].isna().sum())
print("\nUnique companies:", merged["company"].nunique())
print("\nTop 20 companies:")
print(merged["company"].value_counts().head(20))

Missing companies: 0

Unique companies: 274

Top 20 companies:
company
Careerplus                                         37
ictjobs (Stellenmarkt)                             30
progress personal ag                               21
myScience                                          20
CERN European Organization for Nuclear Research    18
Scandit AG                                         15
Avalect Personaldienstleistungen                   15
Joker Personal AG                                  14
Banque Pictet & Cie SA                             14
Hitachi Energy AG                                  13
Bank Julius Bär & Co. AG                           12
On AG                                              11
Personal Knobel AG                                 10
Rocken®                                             9
SonarSource SA                                      9
Randstad (Schweiz) AG                               9
Nexthink SA                                         8
Hamilton Se

## Step 3: Build classifier with keyword matching 

In [17]:
# Sample descriptions from different companies to identify keyword patterns
for i in [0, 10, 20, 50, 100, 200]:
    print(f"--- Row {i} ---")
    print(f"Company: {merged['company'].iloc[i]}")
    print(f"Description (first 300 chars): {merged['description'].iloc[i][:300]}")
    print()

--- Row 0 ---
Company: Academic Work Switzerland
Description (first 300 chars): About the job Tu es actuellement étudiant.e ? Es-tu disponible rapidement pour une mission de 6 mois à 40% ? Maîtrises-tu Power BI ainsi que le tri et le nettoyage de données ? Es-tu reconnu.e pour ton esprit structuré et ta capacité de synthèse ? Mission étudiante 40% : Data Analyst Localisation: R

--- Row 10 ---
Company: HES-SO Valais-Wallis
Description (first 300 chars): About the job The University of Applied Sciences and Arts Western Switzerland Valais (HES-SO Valais-Wallis) has over 2,800 students. Through research and innovation, it contributes to economic and social development and the creation of jobs. The new Energypolis campus, home to the School of Engineer

--- Row 20 ---
Company: Stadler Rail Group
Description (first 300 chars): Job summary Show Stadler offers a wide range of international career opportunities. Join us to shape the mobility of tomorrow! Tasks Manage profitability across the l

## Classifier-info 

To make sure each company matches the right industry codes (BFS NOGA Codes), we need to do the following steps: 

- Step 1: Load the official NOGA code list with descriptions
- Step 2: For each company, search their description for NOGA-related keywords
- Step 3: Map the matched NOGA code to the BFS industry category

In [26]:
# NOGA Code classifier with boundary matching from job descriptions 

import re

# The 17 BFS categories we need to map to
BFS_INDUSTRIES = {
    "62-63 IT services":                        ["software", "saas", "platform", "cybersecurity",
                                                  "tech company", "digital product", "app",
                                                  "sonar", "nexthink", "scandit", "adnovum"],
    "64-66 Finance and Insurance":              ["bank", "banque", "banking", "insurance",
                                                  "versicherung", "asset management", "wealth",
                                                  "investment fund", "trading", "pictet",
                                                  "julius", "ubs", "efg", "comparis"],
    "86-88 Health and social work":             ["hospital", "spital", "klinik", "clinique",
                                                  "pharmaceutical", "pharma", "novartis", "roche",
                                                  "lonza", "sonova", "medical", "healthcare",
                                                  "patients", "clinical"],
    "85 Education":                             ["university", "université", "universität",
                                                  "hochschule", "hes-so", "epfl", "eth zurich",
                                                  "eth zürich", "phd", "postdoc", "faculty",
                                                  "research institute", "academic institution"],
    "10-33 Manufacturing":                      ["manufacturing", "industrial", "factory",
                                                  "production plant", "stadler", "abb", "siemens",
                                                  "hitachi", "schindler", "swatch", "machinery"],
    "41-43 Construction":                       ["construction", "real estate developer",
                                                  "implenia", "holcim", "strabag"],
    "49-53 Transport and storage":              ["railway", "rail", "ferroviaire", "logistics",
                                                  "aviation", "airline", "shipping", "sbb",
                                                  "post group", "dhl", "fedex"],
    "84 Public administration":                 ["federal office", "bundesamt", "government agency",
                                                  "municipality", "cern", "public administration"],
    "68-75 Real estate & professional services":["consulting firm", "management consulting",
                                                  "deloitte", "pwc", "kpmg", "mckinsey",
                                                  "bcg", "accenture", "law firm"],
    "55-56 Hospitality":                        ["hotel", "restaurant", "hospitality", "tourism"],
    "58-63 ICT":                                ["telecom", "telecommunications", "swisscom",
                                                  "sunrise", "salt mobile"],
}

RECRUITERS = ["academic work", "careerplus", "adecco", "randstad", "manpower", "michael page",
              "hays", "rocken", "joker personal", "progress personal", "avalect", "hamilton",
              "personal knobel", "ictjobs", "myscience", "amaris", "mantu"]

def classify_industry(company, description):
    company_l = company.lower()
    desc_l = description.lower()
    
    is_recruiter = any(r in company_l for r in RECRUITERS)
    # Recruiters: search description only; real employers: search both
    text = desc_l if is_recruiter else (company_l + " " + desc_l)
    
    for industry, keywords in BFS_INDUSTRIES.items():
        for kw in keywords:
            # Whole-phrase match with word boundaries
            if re.search(r'\b' + re.escape(kw) + r'\b', text):
                return industry
    
    return "62-63 IT services"  # default fallback for unmatched tech jobs

# Test on the tricky rows
for i in [0, 10, 20, 50, 100, 200]:
    row = merged.iloc[i]
    result = classify_industry(row["company"], row["description"])
    print(f"Row {i} | {row['company'][:35]:<35} → {result}")

Row 0 | Academic Work Switzerland           → 85 Education
Row 10 | HES-SO Valais-Wallis                → 85 Education
Row 20 | Stadler Rail Group                  → 10-33 Manufacturing
Row 50 | Implenia Schweiz AG                 → 62-63 IT services
Row 100 | myScience                           → 62-63 IT services
Row 200 | Deloitte AG                         → 85 Education


In [27]:
# Descriptions don't reveal correct NOGA code for all. Quick diagnostic: 

for i in [0, 10, 50, 200]:
    row = merged.iloc[i]
    print(f"--- Row {i}: {row['company']} ---")
    print(f"Description (first 800 chars): {row['description'][:800]}")
    print()

--- Row 0: Academic Work Switzerland ---
Description (first 800 chars): About the job Tu es actuellement étudiant.e ? Es-tu disponible rapidement pour une mission de 6 mois à 40% ? Maîtrises-tu Power BI ainsi que le tri et le nettoyage de données ? Es-tu reconnu.e pour ton esprit structuré et ta capacité de synthèse ? Mission étudiante 40% : Data Analyst Localisation: Riviera Date de début: Dès que possible Durée: Temps partiel, 40%- 6 mois de mission Type de travail: Mission A propos du rôle Rejoignez notre client, une compagnie ferroviaire en pleine transformation digitale, récompensée pour ses projets innovants. Contribuez à l'optimisation de la gestion des données voyageurs et à la création d'outils décisionnels percutants pour une mission de 6 mois à 40%. En tant que Data Analyst, vous jouerez un rôle clé dans la transformation numérique du département qu

--- Row 10: HES-SO Valais-Wallis ---
Description (first 800 chars): About the job The University of Applied Sciences and Arts 

In [28]:
# Fix wrong company matching

import re

BFS_INDUSTRIES = [  # ORDER MATTERS — most specific first
    ("49-53 Transport and storage",          ["railway", "rail", "ferroviaire", "logistics",
                                               "aviation", "airline", "shipping", "sbb"]),
    ("41-43 Construction",                   ["construction", "implenia", "holcim", "strabag",
                                               "immobiliendienstleister", "bau-"]),
    ("10-33 Manufacturing",                  ["manufacturing", "industrial", "factory",
                                               "stadler", "abb", "siemens", "hitachi",
                                               "schindler", "swatch", "machinery"]),
    ("86-88 Health and social work",         ["hospital", "spital", "klinik", "clinique",
                                               "pharmaceutical", "pharma", "novartis", "roche",
                                               "lonza", "sonova", "medical", "healthcare",
                                               "patients", "clinical", "peptide", "bioconjugation"]),
    ("64-66 Finance and Insurance",          ["bank", "banque", "banking", "insurance",
                                               "versicherung", "asset management", "wealth",
                                               "investment fund", "trading", "pictet",
                                               "julius", "ubs", "efg", "comparis"]),
    ("68-75 Real estate & professional services", ["deloitte", "pwc", "kpmg", "mckinsey",
                                               "bcg", "accenture", "consulting firm",
                                               "management consulting", "law firm"]),
    ("85 Education",                         ["university", "université", "universität",
                                               "hochschule", "hes-so", "epfl", "eth zurich",
                                               "eth zürich", "phd", "postdoc", "faculty",
                                               "research institute"]),
    ("84 Public administration",             ["federal office", "bundesamt", "government agency",
                                               "municipality", "cern", "public administration"]),
    ("55-56 Hospitality",                    ["hotel", "restaurant", "hospitality", "tourism"]),
    ("58-63 ICT",                            ["telecom", "telecommunications", "swisscom",
                                               "sunrise", "salt mobile"]),
    ("62-63 IT services",                    ["software", "saas", "platform", "cybersecurity",
                                               "nexthink", "scandit", "sonar"]),
]

RECRUITERS = ["academic work", "careerplus", "adecco", "randstad", "manpower", "michael page",
              "hays", "rocken", "joker personal", "progress personal", "avalect", "hamilton",
              "personal knobel", "ictjobs", "myscience", "amaris", "mantu"]

def classify_industry(company, description):
    company_l = company.lower()
    desc_l = description.lower()
    is_recruiter = any(r in company_l for r in RECRUITERS)
    text = desc_l if is_recruiter else (company_l + " " + desc_l)

    for industry, keywords in BFS_INDUSTRIES:
        for kw in keywords:
            if re.search(r'\b' + re.escape(kw) + r'\b', text):
                return industry

    return "62-63 IT services"

# Test
for i in [0, 10, 20, 50, 100, 200]:
    row = merged.iloc[i]
    result = classify_industry(row["company"], row["description"])
    print(f"Row {i} | {row['company'][:35]:<35} → {result}")

Row 0 | Academic Work Switzerland           → 49-53 Transport and storage
Row 10 | HES-SO Valais-Wallis                → 10-33 Manufacturing
Row 20 | Stadler Rail Group                  → 49-53 Transport and storage
Row 50 | Implenia Schweiz AG                 → 41-43 Construction
Row 100 | myScience                           → 85 Education
Row 200 | Deloitte AG                         → 68-75 Real estate & professional services


In [29]:
# # Diagnose Row 10 (HES-SO is not Manufacturing) 
row = merged.iloc[10]
text = row["description"].lower()

# Check which manufacturing keyword is matching
mfg_keywords = ["manufacturing", "industrial", "factory", "stadler", "abb", 
                 "siemens", "hitachi", "schindler", "swatch", "machinery"]

for kw in mfg_keywords:
    if re.search(r'\b' + re.escape(kw) + r'\b', text):
        print(f"Matched: '{kw}'")

Matched: 'manufacturing'
Matched: 'industrial'


In [30]:
# Improved Keyword Matching 

BFS_INDUSTRIES = [  # ORDER MATTERS — most specific first
    ("49-53 Transport and storage",          ["railway", "rail", "ferroviaire", "logistics",
                                               "aviation", "airline", "shipping", "sbb"]),
    ("41-43 Construction",                   ["construction", "implenia", "holcim", "strabag",
                                               "immobiliendienstleister", "bau-"]),
    ("10-33 Manufacturing",                  ["stadler", "abb ", "siemens", "hitachi",
                                               "schindler", "swatch", "autoform",
                                               "production facility", "assembly line",
                                               "manufacture of"]),   # removed "manufacturing", "industrial", "factory"
    ("86-88 Health and social work",         ["hospital", "spital", "klinik", "clinique",
                                               "pharmaceutical", "pharma", "novartis", "roche",
                                               "lonza", "sonova", "medical", "healthcare",
                                               "patients", "clinical", "peptide", "bioconjugation"]),
    ("64-66 Finance and Insurance",          ["bank", "banque", "banking", "insurance",
                                               "versicherung", "asset management", "wealth",
                                               "investment fund", "trading", "pictet",
                                               "julius", "ubs", "efg", "comparis"]),
    ("68-75 Real estate & professional services", ["deloitte", "pwc", "kpmg", "mckinsey",
                                               "bcg", "accenture", "consulting firm",
                                               "management consulting", "law firm"]),
    ("85 Education",                         ["university", "université", "universität",
                                               "hochschule", "hes-so", "epfl", "eth zurich",
                                               "eth zürich", "phd", "postdoc", "faculty",
                                               "research institute"]),
    ("84 Public administration",             ["federal office", "bundesamt", "government agency",
                                               "municipality", "cern", "public administration"]),
    ("55-56 Hospitality",                    ["hotel", "restaurant", "hospitality", "tourism"]),
    ("58-63 ICT",                            ["telecom", "telecommunications", "swisscom",
                                               "sunrise", "salt mobile"]),
    ("62-63 IT services",                    ["software", "saas", "platform", "cybersecurity",
                                               "nexthink", "scandit", "sonar"]),
]

# Re-test
for i in [0, 10, 20, 50, 100, 200]:
    row = merged.iloc[i]
    result = classify_industry(row["company"], row["description"])
    print(f"Row {i} | {row['company'][:35]:<35} → {result}")

Row 0 | Academic Work Switzerland           → 49-53 Transport and storage
Row 10 | HES-SO Valais-Wallis                → 86-88 Health and social work
Row 20 | Stadler Rail Group                  → 49-53 Transport and storage
Row 50 | Implenia Schweiz AG                 → 41-43 Construction
Row 100 | myScience                           → 85 Education
Row 200 | Deloitte AG                         → 68-75 Real estate & professional services


In [31]:
# Row 10 still wrong, check company first before description 

def classify_industry(company, description):
    company_l = company.lower()
    desc_l = description.lower()
    is_recruiter = any(r in company_l for r in RECRUITERS)

    if is_recruiter:
        # Recruiters: description only
        search_text = desc_l
    else:
        # Real employers: check company name FIRST, then fall back to description
        for industry, keywords in BFS_INDUSTRIES:
            for kw in keywords:
                if re.search(r'\b' + re.escape(kw) + r'\b', company_l):
                    return industry
        # Company name didn't match — now try description
        search_text = desc_l

    for industry, keywords in BFS_INDUSTRIES:
        for kw in keywords:
            if re.search(r'\b' + re.escape(kw) + r'\b', search_text):
                return industry

    return "62-63 IT services"

# Re-test
for i in [0, 10, 20, 50, 100, 200]:
    row = merged.iloc[i]
    result = classify_industry(row["company"], row["description"])
    print(f"Row {i} | {row['company'][:35]:<35} → {result}")

Row 0 | Academic Work Switzerland           → 49-53 Transport and storage
Row 10 | HES-SO Valais-Wallis                → 85 Education
Row 20 | Stadler Rail Group                  → 49-53 Transport and storage
Row 50 | Implenia Schweiz AG                 → 41-43 Construction
Row 100 | myScience                           → 85 Education
Row 200 | Deloitte AG                         → 68-75 Real estate & professional services


In [32]:
# Run classifier on all 743 rows 

# Run on full dataset
merged["industry"] = merged.apply(
    lambda row: classify_industry(row["company"], row["description"]), axis=1
)

# Check distribution
print("Industry distribution:")
print(merged["industry"].value_counts())
print("\nUnclassified (fallback to IT services):", 
      (merged["industry"] == "62-63 IT services").sum())

# Spot check 10 random rows
sample = merged[["company", "industry"]].sample(10, random_state=42)
print("\nRandom sample:")
print(sample.to_string())

Industry distribution:
industry
62-63 IT services                            345
64-66 Finance and Insurance                   83
86-88 Health and social work                  80
85 Education                                  74
49-53 Transport and storage                   44
41-43 Construction                            36
10-33 Manufacturing                           34
68-75 Real estate & professional services     19
84 Public administration                      18
58-63 ICT                                      7
55-56 Hospitality                              3
Name: count, dtype: int64

Unclassified (fallback to IT services): 345

Random sample:
                                  company                      industry
609                          die Mobiliar             62-63 IT services
539                  Hamilton Services AG  86-88 Health and social work
693                    Personal Knobel AG             62-63 IT services
350                             Bühler AG             

## Quick Diagnostic 

### Problem 1: Many companies are still matched with the wrong NOGA code. 
die Mobiliar → should be Insurance, not IT
Bühler AG → industrial machinery manufacturer, not IT
Varo Refining → oil refining, not Finance
BKW Energie AG → energy company, not IT
Hamilton Services AG → should check (recruiter?)

### Problem 2: IT NOGA Code fallback for unclassified jobs is too high. 
The 345 IT fallbacks lies at 46% — many are real companies that just aren't matching. We add more company-name keywords to catch these.

In [33]:
# See all unique companies falling back to IT services
it_companies = merged[merged["industry"] == "62-63 IT services"]["company"].value_counts()
print(it_companies.head(30))

company
Careerplus                              22
ictjobs (Stellenmarkt)                  22
progress personal ag                    21
Scandit AG                              15
Avalect Personaldienstleistungen        14
Joker Personal AG                       14
On AG                                    9
Personal Knobel AG                       9
Rocken®                                  9
Randstad (Schweiz) AG                    8
Nexthink SA                              8
Prime21                                  6
Infragon Ingenieure AG                   6
Business & Decision (Suisse) SA          5
lastminute.com group                     4
salesforce.com Sàrl                      4
Bühler AG                                4
Integraal Data Services                  4
Trio Personal                            4
BKW Energie AG                           4
enerpeak AG                              4
GetYourGuide AG                          4
Halter AG                                4
Jab

In [34]:
# Improved classifier to solve both problems 

# Add missing recruiters
RECRUITERS = ["academic work", "careerplus", "adecco", "randstad", "manpower", "michael page",
              "hays", "rocken", "joker personal", "progress personal", "avalect", "hamilton",
              "personal knobel", "ictjobs", "myscience", "amaris", "mantu", "trio personal",
              "art of work", "prime21", "integraal"]

# Add missing companies to BFS_INDUSTRIES
BFS_INDUSTRIES = [
    ("49-53 Transport and storage",          ["railway", "rail", "ferroviaire", "logistics",
                                               "aviation", "airline", "shipping", "sbb",
                                               "lastminute", "getyourguide"]),
    ("41-43 Construction",                   ["construction", "implenia", "holcim", "strabag",
                                               "immobiliendienstleister", "halter ag",
                                               "infragon", "frey+gnehm"]),
    ("10-33 Manufacturing",                  ["stadler", "siemens", "hitachi", "schindler",
                                               "swatch", "autoform", "bühler", "buhler",
                                               "jabil", "helbling technik",
                                               "production facility", "manufacture of"]),
    ("86-88 Health and social work",         ["hospital", "spital", "klinik", "clinique",
                                               "pharmaceutical", "pharma", "novartis", "roche",
                                               "lonza", "sonova", "medical", "healthcare",
                                               "patients", "clinical", "peptide"]),
    ("64-66 Finance and Insurance",          ["bank", "banque", "banking", "insurance",
                                               "versicherung", "asset management", "wealth",
                                               "investment fund", "trading", "pictet",
                                               "julius", "ubs", "efg", "comparis",
                                               "mobiliar", "zurich insurance", "axa",
                                               "helvetia", "baloise", "vontobel"]),
    ("68-75 Real estate & professional services", ["deloitte", "pwc", "kpmg", "mckinsey",
                                               "bcg", "accenture", "consulting firm",
                                               "management consulting", "law firm",
                                               "business & decision", "axians"]),
    ("85 Education",                         ["university", "université", "universität",
                                               "hochschule", "hes-so", "epfl", "eth zurich",
                                               "eth zürich", "phd", "postdoc", "faculty",
                                               "research institute", "nccr"]),
    ("84 Public administration",             ["federal office", "bundesamt", "government agency",
                                               "municipality", "cern", "public administration",
                                               "etat du canton", "kanton"]),
    ("55-56 Hospitality",                    ["hotel", "restaurant", "hospitality", "tourism"]),
    ("62-63 IT services",                    ["software", "saas", "cybersecurity", "nexthink",
                                               "scandit", "sonar", "salesforce", "on ag",
                                               "servicenow"]),
    ("58-63 ICT",                            ["telecom", "telecommunications", "swisscom",
                                               "sunrise", "salt mobile", "bkw", "enerpeak",
                                               "infragon"]),
]

# Re-run on full dataset
merged["industry"] = merged.apply(
    lambda row: classify_industry(row["company"], row["description"]), axis=1
)

print("Industry distribution:")
print(merged["industry"].value_counts())
print("\nFallback to IT services:", (merged["industry"] == "62-63 IT services").sum())

# Check remaining fallbacks
it_companies = merged[merged["industry"] == "62-63 IT services"]["company"].value_counts()
print("\nRemaining IT fallbacks:")
print(it_companies.head(20))

Industry distribution:
industry
62-63 IT services                            294
64-66 Finance and Insurance                   86
86-88 Health and social work                  79
85 Education                                  73
49-53 Transport and storage                   50
41-43 Construction                            47
10-33 Manufacturing                           46
84 Public administration                      26
68-75 Real estate & professional services     26
58-63 ICT                                     13
55-56 Hospitality                              3
Name: count, dtype: int64

Fallback to IT services: 294

Remaining IT fallbacks:
company
Careerplus                          22
ictjobs (Stellenmarkt)              21
progress personal ag                21
Scandit AG                          15
Joker Personal AG                   14
Avalect Personaldienstleistungen    13
On AG                               11
Rocken®                              9
Personal Knobel AG          

In [35]:
# Recruiting Companies are not recognised properly. 

# Check why Careerplus isn't being caught as recruiter
test_company = "Careerplus"
print("Is recruiter?", any(r in test_company.lower() for r in RECRUITERS))
print("Company lower:", test_company.lower())
print("Checking against:", [r for r in RECRUITERS if r in test_company.lower()])

Is recruiter? True
Company lower: careerplus
Checking against: ['careerplus']


In [36]:
# For remaining IT companies, check if they are real companies (not recruiters)
# that we should be able to classify better
real_companies_it = merged[
    (merged["industry"] == "62-63 IT services") & 
    (~merged["company"].str.lower().apply(lambda c: any(r in c for r in RECRUITERS)))
]["company"].value_counts()

print("Real companies (non-recruiters) falling back to IT services:")
print(real_companies_it)

Real companies (non-recruiters) falling back to IT services:
company
Scandit AG             15
On AG                  11
Nexthink SA             8
ServiceNow              7
salesforce.com Sàrl     4
                       ..
FMV SA                  1
ALSTOM Schweiz AG       1
Georg Fischer           1
Neho                    1
Fachkraft.ch GmbH       1
Name: count, Length: 92, dtype: int64


In [37]:
# Check companies that are obviously NOT IT
suspicious = ["ALSTOM Schweiz AG", "Georg Fischer", "FMV SA"]

for company in suspicious:
    row = merged[merged["company"] == company].iloc[0]
    print(f"--- {company} ---")
    print(row["description"][:400])
    print()

--- ALSTOM Schweiz AG ---
Job summary Show Join Alstom as a VIE - Methods & Tools Quality Engineer in Villeneuve! Experience a dynamic work environment focused on innovation and sustainability. Tasks Automate quality defect data for efficient data flow. Create dashboards with Power BI for insightful decision-making. Use Machine Learning to predict and cluster train defects. Skills Master’s degree in Applied Mathematics, Da

--- Georg Fischer ---
About the job Software PLC Engineer – R&D 100% Switzerland, Losone Your tasks Participate in the analysis of project needs and contribute to the definition of software requirements, including inputs from Market, Production, and R&D. Collaborate with HMI, process control, mechanical, and electrical teams to define specifications for complex PLC and real-time software components. Develop, test, and 

--- FMV SA ---
About the job Nous sommes la société valaisanne appelée à devenir le leader suisse de la production hydroélectrique. Avec plus de 130

In [38]:
# See full list of companies still misclassified 

real_companies_it = merged[
    (merged["industry"] == "62-63 IT services") & 
    (~merged["company"].str.lower().apply(lambda c: any(r in c for r in RECRUITERS)))
]["company"].value_counts()

print(real_companies_it.to_string())

company
Scandit AG                                       15
On AG                                            11
Nexthink SA                                       8
ServiceNow                                        7
salesforce.com Sàrl                               4
VF International Sagl                             3
Hirslanden AG                                     3
Bechtle Suisse SA                                 3
Infomaniak Network SA                             3
Bobst Mex SA                                      2
yellowshark                                       2
Exeon Analytics AG                                2
INP Schweiz AG                                    2
Philippe Lebert GmbH                              2
Green                                             2
RUAG AG                                           2
selected sa                                       2
Agie Charmilles SA, Losone                        2
HYDRO Exploitation SA                             2
JobT

In [39]:
# Improved classifier 

RECRUITERS = ["academic work", "careerplus", "adecco", "randstad", "manpower", "michael page",
              "hays", "rocken", "joker personal", "progress personal", "avalect", "hamilton",
              "personal knobel", "ictjobs", "myscience", "amaris", "mantu", "trio personal",
              "art of work", "prime21", "integraal", "yellowshark", "jobtalen", "selected sa",
              "finders", "pks personal", "universal-job", "workmanagement", "endago",
              "engineering management selection", "page group", "fachkraft"]

BFS_INDUSTRIES = [
    ("49-53 Transport and storage",          ["railway", "rail", "ferroviaire", "logistics",
                                               "aviation", "airline", "shipping", "sbb",
                                               "lastminute", "getyourguide", "edelweiss air",
                                               "sr technics", "mobility genossenschaft",
                                               "körber supply"]),
    ("41-43 Construction",                   ["construction", "implenia", "holcim", "strabag",
                                               "immobiliendienstleister", "halter ag",
                                               "infragon", "frey+gnehm", "planair"]),
    ("10-33 Manufacturing",                  ["stadler", "siemens", "hitachi", "schindler",
                                               "swatch", "autoform", "bühler", "buhler",
                                               "jabil", "helbling technik", "bobst",
                                               "ruag", "agie charmilles", "mettler-toledo",
                                               "georg fischer", "alstom", "wolfensberger",
                                               "rollomatic", "starrag", "syngenta",
                                               "production facility", "manufacture of"]),
    ("86-88 Health and social work",         ["hospital", "spital", "klinik", "clinique",
                                               "pharmaceutical", "pharma", "novartis", "roche",
                                               "lonza", "sonova", "medical", "healthcare",
                                               "patients", "clinical", "peptide",
                                               "hirslanden", "csl behring", "swica"]),
    ("64-66 Finance and Insurance",          ["bank", "banque", "banking", "insurance",
                                               "versicherung", "asset management", "wealth",
                                               "investment fund", "trading", "pictet",
                                               "julius", "ubs", "efg", "comparis",
                                               "mobiliar", "zurich insurance", "axa",
                                               "helvetia", "baloise", "vontobel",
                                               "kantonalbank", "vaudoise", "groupe mutuel",
                                               "six group", "css versicherung"]),
    ("68-75 Real estate & professional services", ["deloitte", "pwc", "kpmg", "mckinsey",
                                               "bcg", "accenture", "consulting firm",
                                               "management consulting", "law firm",
                                               "business & decision", "axians", "sgs",
                                               "mazars", "adesso", "nielseniq"]),
    ("85 Education",                         ["university", "université", "universität",
                                               "hochschule", "hes-so", "epfl", "eth zurich",
                                               "eth zürich", "phd", "postdoc", "faculty",
                                               "research institute", "nccr", "empa",
                                               "haute ecole"]),
    ("84 Public administration",             ["federal office", "bundesamt", "government agency",
                                               "municipality", "cern", "public administration",
                                               "etat du canton", "kanton"]),
    ("55-56 Hospitality",                    ["hotel", "restaurant", "hospitality", "tourism"]),
    ("58-63 ICT",                            ["telecom", "telecommunications", "swisscom",
                                               "sunrise", "salt mobile", "bkw", "enerpeak",
                                               "hydro exploitation", "fmv sa", "green ",
                                               "peoplefone"]),
    ("62-63 IT services",                    ["software", "saas", "cybersecurity", "nexthink",
                                               "scandit", "sonar", "salesforce", "on ag",
                                               "servicenow", "infomaniak", "kudelski",
                                               "open systems", "infoguard", "ti&m"]),
]

# Re-run on full dataset
merged["industry"] = merged.apply(
    lambda row: classify_industry(row["company"], row["description"]), axis=1
)

print("Industry distribution:")
print(merged["industry"].value_counts())
print("\nFallback to IT services:", (merged["industry"] == "62-63 IT services").sum())

Industry distribution:
industry
62-63 IT services                            255
64-66 Finance and Insurance                   92
86-88 Health and social work                  82
85 Education                                  70
10-33 Manufacturing                           62
49-53 Transport and storage                   52
41-43 Construction                            47
68-75 Real estate & professional services     31
84 Public administration                      26
58-63 ICT                                     23
55-56 Hospitality                              3
Name: count, dtype: int64

Fallback to IT services: 255


In [40]:
# Fix On AG - move from IT to Manufacturing
BFS_INDUSTRIES[2] = ("10-33 Manufacturing", ["stadler", "siemens", "hitachi", "schindler",
                                               "swatch", "autoform", "bühler", "buhler",
                                               "jabil", "helbling technik", "bobst",
                                               "ruag", "agie charmilles", "mettler-toledo",
                                               "georg fischer", "alstom", "wolfensberger",
                                               "rollomatic", "starrag", "syngenta", "on ag",
                                               "production facility", "manufacture of"])

# Re-run
merged["industry"] = merged.apply(
    lambda row: classify_industry(row["company"], row["description"]), axis=1
)

print("Industry distribution:")
print(merged["industry"].value_counts())
print("\nFallback to IT services:", (merged["industry"] == "62-63 IT services").sum())

# Verify On AG
print("\nOn AG classification:", merged[merged["company"] == "On AG"]["industry"].unique())

Industry distribution:
industry
62-63 IT services                            244
64-66 Finance and Insurance                   92
86-88 Health and social work                  82
10-33 Manufacturing                           73
85 Education                                  70
49-53 Transport and storage                   52
41-43 Construction                            47
68-75 Real estate & professional services     31
84 Public administration                      26
58-63 ICT                                     23
55-56 Hospitality                              3
Name: count, dtype: int64

Fallback to IT services: 244

On AG classification: ['10-33 Manufacturing']


In [44]:
# Thorough check how many companies are still misclassified 

# Re-run the full pipeline from scratch
import pandas as pd
import re

# Load data
merged = pd.read_csv("../data/processed/jobs_merged_final.csv")
macro_industry = pd.read_csv("../notebooks/data/external/bfs_vacancies_economic_division_clean.csv")

# --- Classifier ---
RECRUITERS = ["academic work", "careerplus", "adecco", "randstad", "manpower", "michael page",
              "hays", "rocken", "joker personal", "progress personal", "avalect", "hamilton",
              "personal knobel", "ictjobs", "myscience", "amaris", "mantu", "trio personal",
              "art of work", "prime21", "integraal", "yellowshark", "jobtalent", "selected sa",
              "finders", "pks personal", "universal-job", "workmanagement", "endago",
              "engineering management selection", "page group", "fachkraft"]

BFS_INDUSTRIES = [
    ("49-53 Transport and storage",          ["railway", "rail", "ferroviaire", "logistics",
                                               "aviation", "airline", "shipping", "sbb",
                                               "lastminute", "getyourguide", "edelweiss air",
                                               "sr technics", "mobility genossenschaft",
                                               "körber supply"]),
    ("41-43 Construction",                   ["construction", "implenia", "holcim", "strabag",
                                               "immobiliendienstleister", "halter ag",
                                               "infragon", "frey+gnehm", "planair"]),
    ("10-33 Manufacturing",                  ["stadler", "siemens", "hitachi", "schindler",
                                               "swatch", "autoform", "bühler", "buhler",
                                               "jabil", "helbling technik", "bobst",
                                               "ruag", "agie charmilles", "mettler-toledo",
                                               "georg fischer", "alstom", "wolfensberger",
                                               "rollomatic", "starrag", "syngenta", "on ag",
                                               "production facility", "manufacture of"]),
    ("86-88 Health and social work",         ["hospital", "spital", "klinik", "clinique",
                                               "pharmaceutical", "pharma", "novartis", "roche",
                                               "lonza", "sonova", "medical", "healthcare",
                                               "patients", "clinical", "peptide",
                                               "hirslanden", "csl behring", "swica"]),
    ("64-66 Finance and Insurance",          ["bank", "banque", "banking", "insurance",
                                               "versicherung", "asset management", "wealth",
                                               "investment fund", "trading", "pictet",
                                               "julius", "ubs", "efg", "comparis",
                                               "mobiliar", "zurich insurance", "axa",
                                               "helvetia", "baloise", "vontobel",
                                               "kantonalbank", "vaudoise", "groupe mutuel",
                                               "six group", "css"]),
    ("68-75 Real estate & professional services", ["deloitte", "pwc", "kpmg", "mckinsey",
                                               "bcg", "accenture", "consulting firm",
                                               "management consulting", "law firm",
                                               "business & decision", "axians", "sgs",
                                               "mazars", "adesso", "nielseniq"]),
    ("85 Education",                         ["university", "université", "universität",
                                               "hochschule", "hes-so", "epfl", "eth zurich",
                                               "eth zürich", "phd", "postdoc", "faculty",
                                               "research institute", "nccr", "empa",
                                               "haute ecole"]),
    ("84 Public administration",             ["federal office", "bundesamt", "government agency",
                                               "municipality", "cern", "public administration",
                                               "etat du canton", "kanton"]),
    ("55-56 Hospitality",                    ["hotel", "restaurant", "hospitality", "tourism"]),
    ("58-63 ICT",                            ["telecom", "telecommunications", "swisscom",
                                               "sunrise", "salt mobile", "bkw", "enerpeak",
                                               "hydro exploitation", "fmv sa", "green ",
                                               "peoplefone"]),
    ("62-63 IT services",                    ["software", "saas", "cybersecurity", "nexthink",
                                               "scandit", "sonar", "salesforce", "on ag",
                                               "servicenow", "infomaniak", "kudelski",
                                               "open systems", "infoguard", "ti&m"]),
]

def classify_industry(company, description):
    company_l = company.lower()
    desc_l = description.lower()
    is_recruiter = any(r in company_l for r in RECRUITERS)

    if is_recruiter:
        search_text = desc_l
    else:
        for industry, keywords in BFS_INDUSTRIES:
            for kw in keywords:
                if re.search(r'\b' + re.escape(kw) + r'\b', company_l):
                    return industry
        search_text = desc_l

    for industry, keywords in BFS_INDUSTRIES:
        for kw in keywords:
            if re.search(r'\b' + re.escape(kw) + r'\b', search_text):
                return industry

    return "62-63 IT services"

# Re-classify
merged["industry"] = merged.apply(
    lambda row: classify_industry(row["company"], row["description"]), axis=1
)

# Now show grouped by industry
company_industry = (merged.groupby(["company", "industry"])
                    .size()
                    .reset_index(name="count")
                    .sort_values(["industry", "count"], ascending=[True, False]))

for industry, group in company_industry.groupby("industry"):
    print(f"\n{'='*60}")
    print(f"  {industry}")
    print(f"{'='*60}")
    print(group[["company", "count"]].to_string(index=False))


  10-33 Manufacturing
                             company  count
                   Hitachi Energy AG     13
                               On AG     11
                      SonarSource SA      9
Jabil Switzerland Manufacturing GmbH      5
                           Bühler AG      4
                        Bobst Mex SA      3
                    Helbling Technik      3
                             RUAG AG      3
          Agie Charmilles SA, Losone      2
                        BOBST MEX SA      2
                  Starrag Vuadens SA      2
                   ALSTOM Schweiz AG      1
           AutoForm Development GmbH      1
                          Careerplus      1
                       Georg Fischer      1
                 Mettler-Toledo GmbH      1
                  Personal Knobel AG      1
                     Pharmatronic AG      1
               Randstad (Schweiz) AG      1
                       Rollomatic SA      1
                           SWISSto12      1
         

In [45]:
# Improved classifier for last remaining anomalies 

# Add more recruiters
RECRUITERS = ["academic work", "careerplus", "adecco", "randstad", "manpower", "michael page",
              "hays", "rocken", "joker personal", "progress personal", "avalect", "hamilton services",
              "personal knobel", "ictjobs", "myscience", "amaris", "mantu", "trio personal",
              "art of work", "prime21", "integraal", "yellowshark", "jobtalent", "selected sa",
              "finders", "pks personal", "universal-job", "workmanagement", "endago",
              "engineering management selection", "page group", "fachkraft", "nemensis",
              "manpower"]

BFS_INDUSTRIES = [
    ("49-53 Transport and storage",          ["railway", "rail", "ferroviaire", "logistics",
                                               "aviation", "airline", "shipping", "sbb",
                                               "lastminute", "getyourguide", "edelweiss air",
                                               "sr technics", "mobility genossenschaft",
                                               "körber supply", "transports publics",
                                               "skycell", "daedalean"]),
    ("41-43 Construction",                   ["construction", "implenia", "holcim", "strabag",
                                               "halter ag", "infragon", "frey+gnehm",
                                               "planair", "ramboll", "bolliger",
                                               "dietziker", "blättler", "bächtold"]),
    ("10-33 Manufacturing",                  ["stadler", "siemens", "hitachi", "schindler",
                                               "swatch", "autoform", "bühler", "buhler",
                                               "jabil", "helbling technik", "bobst",
                                               "ruag", "agie charmilles", "mettler-toledo",
                                               "georg fischer", "alstom", "wolfensberger",
                                               "rollomatic", "starrag", "syngenta", "on ag",
                                               "liebherr", "sika ag", "wingtra",
                                               "winterthur gas", "medmix", "aeschlimann",
                                               "schleuniger", "sensirion", "viasat antenna",
                                               "sonceboz", "gf machining", "büchi labor",
                                               "novotec", "beyond gravity", "varo refining",
                                               "tecan", "zegna", "uhrteil", "vat vakuum",
                                               "swissto12", "pharmatronic",
                                               "production facility", "manufacture of"]),
    ("86-88 Health and social work",         ["hospital", "spital", "klinik", "clinique",
                                               "pharmaceutical", "pharma", "novartis", "roche",
                                               "lonza", "sonova", "medical", "healthcare",
                                               "patients", "clinical", "peptide",
                                               "hirslanden", "csl behring", "swica",
                                               "galenica", "dentsply", "universitätsspital",
                                               "hug -", "solothurner spitäler", "chuv",
                                               "ärztekasse", "bachem", "ucb-pharma",
                                               "thermo fisher", "medartis", "philochem",
                                               "trb chemedica", "eurofins", "abbott",
                                               "thoratec", "varian medical", "zuripharma",
                                               "octapharma", "insel gruppe"]),
    ("64-66 Finance and Insurance",          ["bank", "banque", "banking", "insurance",
                                               "versicherung", "asset management", "wealth",
                                               "investment fund", "pictet", "julius",
                                               "ubs", "efg", "comparis", "mobiliar",
                                               "axa", "helvetia", "baloise", "vontobel",
                                               "kantonalbank", "vaudoise", "groupe mutuel",
                                               "css", "swissquote", "avaloq", "lombard odier",
                                               "scor", "ficoba", "ppcmetrics", "wir bank",
                                               "bank cler", "bank frick", "prismalife",
                                               "orion rechtsschutz", "lgt bank", "finnova",
                                               "swissquant", "kepler cheuvreux"]),
    ("68-75 Real estate & professional services", ["deloitte", "pwc", "kpmg", "mckinsey",
                                               "bcg", "accenture", "business & decision",
                                               "axians", "sgs", "mazars", "adesso",
                                               "nielseniq", "magellan", "ernst & young",
                                               "impact hub", "iss schweiz", "zühlke"]),
    ("85 Education",                         ["university", "université", "universität",
                                               "hochschule", "hes-so", "epfl", "eth zurich",
                                               "eth zürich", "phd", "postdoc", "faculty",
                                               "research institute", "nccr", "empa",
                                               "haute ecole", "eawag", "paul scherrer",
                                               "supercomputing systems"]),
    ("84 Public administration",             ["federal office", "bundesamt", "government agency",
                                               "municipality", "cern", "public administration",
                                               "etat du canton", "kanton", "stadtwerke",
                                               "fia -", "fédération internationale"]),
    ("55-56 Hospitality",                    ["hotel", "restaurant", "hospitality", "tourism"]),
    ("58-63 ICT",                            ["telecom", "telecommunications", "swisscom",
                                               "sunrise", "salt mobile", "bkw", "enerpeak",
                                               "hydro exploitation", "fmv sa", "peoplefone",
                                               "axpo", "energie wasser", "comfone", "green "]),
    ("62-63 IT services",                    ["software", "saas", "cybersecurity", "nexthink",
                                               "scandit", "sonar", "salesforce", "servicenow",
                                               "infomaniak", "kudelski", "open systems",
                                               "infoguard", "ti&m", "frontify", "qoqa",
                                               "jobcloud", "ch media", "bison schweiz",
                                               "speedcom", "exeon", "dexion"]),
]

# Re-run classifier
merged["industry"] = merged.apply(
    lambda row: classify_industry(row["company"], row["description"]), axis=1
)

# Show updated distribution
print("Industry distribution:")
print(merged["industry"].value_counts())

# Show grouped by industry again
company_industry = (merged.groupby(["company", "industry"])
                    .size()
                    .reset_index(name="count")
                    .sort_values(["industry", "count"], ascending=[True, False]))

for industry, group in company_industry.groupby("industry"):
    print(f"\n{'='*60}")
    print(f"  {industry}")
    print(f"{'='*60}")
    print(group[["company", "count"]].to_string(index=False))

Industry distribution:
industry
62-63 IT services                            241
10-33 Manufacturing                           93
64-66 Finance and Insurance                   89
86-88 Health and social work                  75
85 Education                                  66
49-53 Transport and storage                   46
41-43 Construction                            41
68-75 Real estate & professional services     36
58-63 ICT                                     29
84 Public administration                      27
Name: count, dtype: int64

  10-33 Manufacturing
                             company  count
                   Hitachi Energy AG     13
                               On AG     11
                      SonarSource SA      9
Jabil Switzerland Manufacturing GmbH      5
                           Bühler AG      4
                        Bobst Mex SA      3
                    Helbling Technik      3
                             RUAG AG      3
          Agie Charmilles SA, Los

## Quick Diagnostic

Classifier still needs improvement. Quick overview of a few anomalies: 

- SonarSource SA → IT services (not manufacturing — they make code quality software)
- Careerplus, Randstad, Personal Knobel, ictjobs, Hamilton Services, nemensis, myScience appearing in wrong industries — recruiter matching issue
- HUG - Hôpitaux Universitaires → Health (not construction)
- Swiss Life AG → Finance (not construction)
- Kepler Cheuvreux → Finance (already there)
- SIX Group AG appearing in both Finance AND Education — duplicate
- Vitol → Finance/Energy (oil trading), not Education
- Manor AG → Retail (not Education)
- Lenz & Staehelin → Professional services (law firm)
- Büchi Labortechnik → Manufacturing (lab instruments)
- SAPCO SA → check what they do
- isolutions AG → IT services
- Bechtle Suisse SA → IT services
- Energy Vault SA → ICT/Energy
- Huntsman Advanced Materials → Manufacturing
- Maag Pump Systems → Manufacturing
- FIS Switzerland → Finance (financial software)

In [47]:
# Fixed classifier 

# Fix SonarSource - remove from manufacturing, add to IT
# Fix recruiter matching - make sure all recruiters are caught
RECRUITERS = ["academic work", "careerplus", "adecco", "randstad", "manpower", "michael page",
              "hays", "rocken", "joker personal", "progress personal", "avalect", "hamilton services",
              "personal knobel", "ictjobs", "myscience", "amaris", "mantu", "trio personal",
              "art of work", "prime21", "integraal", "yellowshark", "jobtalent", "selected sa",
              "finders", "pks personal", "universal-job", "workmanagement", "endago",
              "engineering management selection", "page group", "fachkraft", "nemensis",
              "manpower", "work selection"]

BFS_INDUSTRIES = [
    ("49-53 Transport and storage",          ["railway", "rail", "ferroviaire", "logistics",
                                               "aviation", "airline", "shipping", "sbb",
                                               "lastminute", "getyourguide", "edelweiss air",
                                               "sr technics", "mobility genossenschaft",
                                               "körber supply", "transports publics",
                                               "skycell", "daedalean", "sihltal"]),
    ("41-43 Construction",                   ["construction", "implenia", "holcim", "strabag",
                                               "halter ag", "infragon", "frey+gnehm",
                                               "planair", "ramboll", "bolliger",
                                               "dietziker", "blättler", "bächtold",
                                               "gritec", "rubitec", "jag jakob",
                                               "officine ghidoni", "specitec", "nbg ingenieure",
                                               "mp ingénieurs"]),
    ("10-33 Manufacturing",                  ["stadler", "siemens", "hitachi", "schindler",
                                               "swatch", "autoform", "bühler", "buhler",
                                               "jabil", "helbling technik", "bobst",
                                               "ruag", "agie charmilles", "mettler-toledo",
                                               "georg fischer", "alstom", "wolfensberger",
                                               "rollomatic", "starrag", "syngenta", "on ag",
                                               "liebherr", "sika ag", "wingtra",
                                               "winterthur gas", "medmix", "aeschlimann",
                                               "schleuniger", "sensirion", "viasat antenna",
                                               "sonceboz", "gf machining", "büchi labor",
                                               "novotec", "beyond gravity", "varo refining",
                                               "tecan", "zegna", "uhrteil", "vat vakuum",
                                               "swissto12", "pharmatronic", "huntsman",
                                               "maag pump", "fisba"]),
    ("86-88 Health and social work",         ["hospital", "spital", "klinik", "clinique",
                                               "pharmaceutical", "pharma", "novartis", "roche",
                                               "lonza", "sonova", "medical", "healthcare",
                                               "patients", "clinical", "peptide",
                                               "hirslanden", "csl behring", "swica",
                                               "galenica", "dentsply", "universitätsspital",
                                               "hug -", "solothurner spitäler", "chuv",
                                               "ärztekasse", "bachem", "ucb-pharma",
                                               "thermo fisher", "medartis", "philochem",
                                               "trb chemedica", "eurofins", "abbott",
                                               "thoratec", "varian medical", "zuripharma",
                                               "octapharma", "insel gruppe", "fkg dentaire",
                                               "celerion", "axepta", "unity schweiz",
                                               "apothekerverband"]),
    ("64-66 Finance and Insurance",          ["bank", "banque", "banking", "insurance",
                                               "versicherung", "asset management", "wealth",
                                               "investment fund", "pictet", "julius",
                                               "ubs", "efg", "comparis", "mobiliar",
                                               "axa", "helvetia", "baloise", "vontobel",
                                               "kantonalbank", "vaudoise", "groupe mutuel",
                                               "css", "swissquote", "avaloq", "lombard odier",
                                               "scor", "ficoba", "ppcmetrics", "wir bank",
                                               "bank cler", "bank frick", "prismalife",
                                               "orion rechtsschutz", "lgt bank", "finnova",
                                               "swissquant", "kepler cheuvreux", "swiss life",
                                               "vitol", "six group", "fis (switzerland)",
                                               "qim info", "stellenwerk", "anansi solutions"]),
    ("68-75 Real estate & professional services", ["deloitte", "pwc", "kpmg", "mckinsey",
                                               "bcg", "accenture", "business & decision",
                                               "axians", "sgs", "mazars", "adesso",
                                               "nielseniq", "magellan", "ernst & young",
                                               "impact hub", "iss schweiz", "zühlke",
                                               "lenz & staehelin", "isolutions",
                                               "bechtle", "lb&partners", "d one value",
                                               "it advanced consulting", "mgי consultants",
                                               "talan", "inp schweiz", "smg swiss"]),
    ("85 Education",                         ["university", "université", "universität",
                                               "hochschule", "hes-so", "epfl", "eth zurich",
                                               "eth zürich", "phd", "postdoc", "faculty",
                                               "research institute", "nccr", "empa",
                                               "haute ecole", "eawag", "paul scherrer",
                                               "supercomputing systems", "sib institut",
                                               "zürcher hochschule"]),
    ("84 Public administration",             ["federal office", "bundesamt", "government agency",
                                               "municipality", "cern", "public administration",
                                               "etat du canton", "kanton", "stadtwerke",
                                               "fédération internationale", "kantonale verwaltung"]),
    ("55-56 Hospitality",                    ["hotel", "restaurant", "hospitality", "tourism",
                                               "manor ag", "vilebrequin", "beldona",
                                               "michael kors", "vf international"]),
    ("58-63 ICT",                            ["telecom", "telecommunications", "swisscom",
                                               "sunrise", "salt mobile", "bkw", "enerpeak",
                                               "hydro exploitation", "fmv sa", "peoplefone",
                                               "axpo", "energie wasser", "comfone", "green ",
                                               "energy vault", "spie schweiz"]),
    ("62-63 IT services",                    ["sonar", "nexthink", "scandit", "salesforce",
                                               "servicenow", "infomaniak", "kudelski",
                                               "open systems", "infoguard", "ti&m", "frontify",
                                               "qoqa", "jobcloud", "ch media", "bison schweiz",
                                               "speedcom", "exeon", "dexion", "extendis",
                                               "nexplore", "at rete", "cross-ing", "sigmalis",
                                               "inera sa", "mgي consultants", "systéa",
                                               "amag group", "enableme", "peoplefone",
                                               "foodtechscout", "sapco", "php lebert",
                                               "yellowshark"]),
]

# Re-run classifier
merged["industry"] = merged.apply(
    lambda row: classify_industry(row["company"], row["description"]), axis=1
)

# Show distribution
print("Industry distribution:")
print(merged["industry"].value_counts())

# Show grouped
company_industry = (merged.groupby(["company", "industry"])
                    .size()
                    .reset_index(name="count")
                    .sort_values(["industry", "count"], ascending=[True, False]))

for industry, group in company_industry.groupby("industry"):
    print(f"\n{'='*60}")
    print(f"  {industry}")
    print(f"{'='*60}")
    print(group[["company", "count"]].to_string(index=False))

Industry distribution:
industry
62-63 IT services                            227
64-66 Finance and Insurance                   95
10-33 Manufacturing                           95
86-88 Health and social work                  70
85 Education                                  60
68-75 Real estate & professional services     50
49-53 Transport and storage                   42
41-43 Construction                            40
58-63 ICT                                     29
84 Public administration                      28
55-56 Hospitality                              7
Name: count, dtype: int64

  10-33 Manufacturing
                                       company  count
                             Hitachi Energy AG     13
                                         On AG     11
                                SonarSource SA      9
          Jabil Switzerland Manufacturing GmbH      5
                                     Bühler AG      4
                                  Bobst Mex SA      3
  

In [48]:
# last fixed classifier 

RECRUITERS = ["academic work", "careerplus", "adecco", "randstad", "manpower", "michael page",
              "hays", "rocken", "joker personal", "progress personal", "avalect", "hamilton services",
              "personal knobel", "ictjobs", "myscience", "amaris", "mantu", "trio personal",
              "art of work", "prime21", "integraal", "yellowshark", "jobtalent", "selected sa",
              "finders", "pks personal", "universal-job", "workmanagement", "endago",
              "engineering management selection", "page group", "fachkraft", "nemensis",
              "manpower", "work selection"]

BFS_INDUSTRIES = [
    ("49-53 Transport and storage",          ["railway", "rail", "ferroviaire",
                                               "aviation", "airline", "shipping", "sbb",
                                               "lastminute", "getyourguide", "edelweiss air",
                                               "sr technics", "mobility genossenschaft",
                                               "körber supply", "transports publics",
                                               "skycell", "sihltal",
                                               "swiss international air lines"]),
    ("41-43 Construction",                   ["implenia", "holcim", "strabag",
                                               "halter ag", "infragon", "frey+gnehm",
                                               "planair", "ramboll", "bolliger",
                                               "dietziker", "blättler", "bächtold",
                                               "gritec", "rubitec", "jag jakob",
                                               "officine ghidoni", "specitec",
                                               "mp ingénieurs"]),
    ("10-33 Manufacturing",                  ["stadler", "siemens", "hitachi", "schindler",
                                               "swatch", "autoform", "bühler", "buhler",
                                               "jabil", "helbling technik", "bobst",
                                               "ruag", "agie charmilles", "mettler-toledo",
                                               "georg fischer", "alstom", "wolfensberger",
                                               "rollomatic", "starrag", "syngenta", "on ag",
                                               "liebherr", "sika ag", "wingtra",
                                               "winterthur gas", "medmix", "aeschlimann",
                                               "schleuniger", "sensirion", "viasat antenna",
                                               "sonceboz", "gf machining", "büchi labor",
                                               "novotec", "beyond gravity", "varo refining",
                                               "tecan", "zegna", "uhrteil", "vat vakuum",
                                               "swissto12", "pharmatronic", "huntsman",
                                               "maag pump", "fisba", "vf international",
                                               "beldona", "michael kors", "vilebrequin",
                                               "apco technologies", "h55", "daedalean"]),
    ("86-88 Health and social work",         ["hospital", "spital", "klinik", "clinique",
                                               "pharmaceutical", "pharma", "novartis", "roche",
                                               "lonza", "sonova", "medical", "healthcare",
                                               "patients", "clinical", "peptide",
                                               "hirslanden", "csl behring", "swica",
                                               "galenica", "dentsply", "universitätsspital",
                                               "hôpitaux universitaires", "solothurner spitäler",
                                               "chuv", "ärztekasse", "bachem", "ucb-pharma",
                                               "thermo fisher", "medartis", "philochem",
                                               "trb chemedica", "eurofins", "abbott",
                                               "thoratec", "varian medical", "zuripharma",
                                               "octapharma", "insel gruppe", "fkg dentaire",
                                               "celerion", "axepta", "unity schweiz",
                                               "apothekerverband"]),
    ("64-66 Finance and Insurance",          ["bank", "banque", "banking", "insurance",
                                               "versicherung", "asset management", "wealth",
                                               "investment fund", "pictet", "julius",
                                               "ubs", "efg", "comparis", "mobiliar",
                                               "axa", "helvetia", "baloise", "vontobel",
                                               "kantonalbank", "vaudoise", "groupe mutuel",
                                               "css", "swissquote", "avaloq", "lombard odier",
                                               "scor", "ficoba", "ppcmetrics", "wir bank",
                                               "bank cler", "bank frick", "prismalife",
                                               "orion rechtsschutz", "lgt bank", "finnova",
                                               "swissquant", "kepler cheuvreux", "swiss life",
                                               "vitol", "six group", "fis (switzerland)",
                                               "qim info", "stellenwerk", "anansi solutions"]),
    ("68-75 Real estate & professional services", ["deloitte", "pwc", "kpmg", "mckinsey",
                                               "bcg", "accenture", "business & decision",
                                               "axians", "sgs", "mazars", "adesso",
                                               "nielseniq", "magellan", "ernst & young",
                                               "impact hub", "iss schweiz", "zühlke",
                                               "lenz & staehelin", "isolutions",
                                               "bechtle", "lb&partners", "d one value",
                                               "it advanced consulting", "talan",
                                               "inp schweiz", "smg swiss", "neho",
                                               "nbg ingenieure"]),
    ("85 Education",                         ["university", "université", "universität",
                                               "hochschule", "hes-so", "epfl", "eth zurich",
                                               "eth zürich", "phd", "postdoc", "faculty",
                                               "research institute", "nccr", "empa",
                                               "haute ecole", "eawag", "paul scherrer",
                                               "supercomputing systems", "sib institut",
                                               "zürcher hochschule", "cern"]),
    ("84 Public administration",             ["federal office", "bundesamt", "government agency",
                                               "municipality", "public administration",
                                               "etat du canton", "kanton", "stadtwerke",
                                               "fédération internationale", "kantonale verwaltung",
                                               # Energy/utilities → mapped here
                                               "bkw", "enerpeak", "hydro exploitation",
                                               "fmv sa", "axpo", "energie wasser",
                                               "energy vault", "spie schweiz", "green "]),
    ("58-63 ICT",                            ["telecom", "telecommunications", "swisscom",
                                               "sunrise", "salt mobile", "peoplefone",
                                               "comfone"]),
    ("62-63 IT services",                    ["SonarSource", "nexthink", "scandit", "salesforce",
                                               "servicenow", "infomaniak", "kudelski",
                                               "open systems", "infoguard", "ti&m", "frontify",
                                               "qoqa", "ch media", "bison schweiz",
                                               "speedcom", "exeon", "dexion", "extendis",
                                               "nexplore", "at rete", "cross-ing", "sigmalis",
                                               "inera sa", "systéa", "amag group", "enableme",
                                               "foodtechscout", "sapco", "philippe lebert",
                                               "manor ag", "daedalean"]),
]

# Re-run classifier
merged["industry"] = merged.apply(
    lambda row: classify_industry(row["company"], row["description"]), axis=1
)

print("Industry distribution:")
print(merged["industry"].value_counts())

# Show grouped
company_industry = (merged.groupby(["company", "industry"])
                    .size()
                    .reset_index(name="count")
                    .sort_values(["industry", "count"], ascending=[True, False]))

for industry, group in company_industry.groupby("industry"):
    print(f"\n{'='*60}")
    print(f"  {industry}")
    print(f"{'='*60}")
    print(group[["company", "count"]].to_string(index=False))

Industry distribution:
industry
62-63 IT services                            233
10-33 Manufacturing                          107
64-66 Finance and Insurance                   94
85 Education                                  79
86-88 Health and social work                  73
68-75 Real estate & professional services     52
49-53 Transport and storage                   36
84 Public administration                      31
41-43 Construction                            30
58-63 ICT                                      8
Name: count, dtype: int64

  10-33 Manufacturing
                                       company  count
                             Hitachi Energy AG     13
                                         On AG     11
                                SonarSource SA      9
          Jabil Switzerland Manufacturing GmbH      5
                                     Bühler AG      4
                                  Daedalean AG      4
                                  Bobst Mex SA     

In [51]:
# Diagnose remaining companies that still mismatch 

# 5 Companies in question
for company in ["SonarSource SA", "Daedalean AG", "Büchi Labortechnik AG", 
                "Supercomputing Systems AG", "VAT Vakuumventile AG"]:
    company_l = company.lower()
    for industry, keywords in BFS_INDUSTRIES:
        for kw in keywords:
            if re.search(r'\b' + re.escape(kw) + r'\b', company_l):
                print(f"{company} → matched '{kw}' → {industry}")
                break

Daedalean AG → matched 'daedalean' → 10-33 Manufacturing
Daedalean AG → matched 'daedalean' → 62-63 IT services
Supercomputing Systems AG → matched 'supercomputing systems' → 85 Education


In [52]:
# Fix 1: Remove daedalean from Manufacturing (it's aviation AI software)
# Fix 2: Move sonar to IT services company-name list (SonarSource) correctly
# Fix 3: Add büchi to Manufacturing (lab instruments)
# Fix 4: Remove supercomputing systems from Education → let it fall to IT services
# Fix 5: Add vat vakuum explicitly to Manufacturing
# Fix 6: Add fis (switzerland) to Finance keywords

# Just update these specific lists:

# Manufacturing - add büchi, remove daedalean
BFS_INDUSTRIES[2] = ("10-33 Manufacturing", ["stadler", "siemens", "hitachi", "schindler",
                                               "swatch", "autoform", "bühler", "buhler",
                                               "jabil", "helbling technik", "bobst",
                                               "ruag", "agie charmilles", "mettler-toledo",
                                               "georg fischer", "alstom", "wolfensberger",
                                               "rollomatic", "starrag", "syngenta", "on ag",
                                               "liebherr", "sika ag", "wingtra",
                                               "winterthur gas", "medmix", "aeschlimann",
                                               "schleuniger", "sensirion", "viasat antenna",
                                               "sonceboz", "gf machining", "büchi labor",
                                               "büchi labortechnik", "novotec", "beyond gravity",
                                               "varo refining", "tecan", "zegna", "uhrteil",
                                               "vat vakuum", "swissto12", "pharmatronic",
                                               "huntsman", "maag pump", "fisba",
                                               "vf international", "beldona", "michael kors",
                                               "vilebrequin", "apco technologies", "h55"])

# Education - remove supercomputing systems
BFS_INDUSTRIES[6] = ("85 Education", ["university", "université", "universität",
                                       "hochschule", "hes-so", "epfl", "eth zurich",
                                       "eth zürich", "phd", "postdoc", "faculty",
                                       "research institute", "nccr", "empa",
                                       "haute ecole", "eawag", "paul scherrer",
                                       "sib institut", "zürcher hochschule", "cern"])

# IT services - add sonar (SonarSource), supercomputing, daedalean
BFS_INDUSTRIES[9] = ("62-63 IT services", ["sonar", "nexthink", "scandit", "salesforce",
                                             "servicenow", "infomaniak", "kudelski",
                                             "open systems", "infoguard", "ti&m", "frontify",
                                             "qoqa", "ch media", "bison schweiz",
                                             "speedcom", "exeon", "dexion", "extendis",
                                             "nexplore", "at rete", "cross-ing", "sigmalis",
                                             "inera sa", "systéa", "amag group", "enableme",
                                             "foodtechscout", "sapco", "philippe lebert",
                                             "manor ag", "daedalean", "supercomputing systems"])

# Finance - add fis (switzerland)
BFS_INDUSTRIES[4] = ("64-66 Finance and Insurance", ["bank", "banque", "banking", "insurance",
                                               "versicherung", "asset management", "wealth",
                                               "investment fund", "pictet", "julius",
                                               "ubs", "efg", "comparis", "mobiliar",
                                               "axa", "helvetia", "baloise", "vontobel",
                                               "kantonalbank", "vaudoise", "groupe mutuel",
                                               "css", "swissquote", "avaloq", "lombard odier",
                                               "scor", "ficoba", "ppcmetrics", "wir bank",
                                               "bank cler", "bank frick", "prismalife",
                                               "orion rechtsschutz", "lgt bank", "finnova",
                                               "swissquant", "kepler cheuvreux", "swiss life",
                                               "vitol", "six group", "fis (switzerland)",
                                               "qim info", "stellenwerk", "anansi solutions"])

# Re-run
merged["industry"] = merged.apply(
    lambda row: classify_industry(row["company"], row["description"]), axis=1
)

# Verify specific companies
for company in ["SonarSource SA", "Daedalean AG", "Büchi Labortechnik AG",
                "Supercomputing Systems AG", "VAT Vakuumventile AG",
                "FIS (Switzerland) SA, Zweigniederlassung Zürich"]:
    rows = merged[merged["company"] == company]
    if len(rows) > 0:
        print(f"{company} → {rows['industry'].iloc[0]}")

print("\nIndustry distribution:")
print(merged["industry"].value_counts())

SonarSource SA → 10-33 Manufacturing
Daedalean AG → 62-63 IT services
Büchi Labortechnik AG → 10-33 Manufacturing
Supercomputing Systems AG → 62-63 IT services
VAT Vakuumventile AG → 62-63 IT services
FIS (Switzerland) SA, Zweigniederlassung Zürich → 86-88 Health and social work

Industry distribution:
industry
62-63 IT services                            238
10-33 Manufacturing                          104
64-66 Finance and Insurance                   94
85 Education                                  77
86-88 Health and social work                  73
68-75 Real estate & professional services     52
49-53 Transport and storage                   36
84 Public administration                      31
41-43 Construction                            30
58-63 ICT                                      8
Name: count, dtype: int64


In [53]:
# Check last anomalies
for company in ["SonarSource SA", "VAT Vakuumventile AG", 
                "FIS (Switzerland) SA, Zweigniederlassung Zürich"]:
    company_l = company.lower()
    print(f"\n{company} (lower: '{company_l}')")
    for industry, keywords in BFS_INDUSTRIES:
        for kw in keywords:
            if re.search(r'\b' + re.escape(kw) + r'\b', company_l):
                print(f"  → matched '{kw}' in {industry}")


SonarSource SA (lower: 'sonarsource sa')

VAT Vakuumventile AG (lower: 'vat vakuumventile ag')

FIS (Switzerland) SA, Zweigniederlassung Zürich (lower: 'fis (switzerland) sa, zweigniederlassung zürich')


In [54]:
# None of them are matching any company-name keyword
# Detailed fix 

# Fix 1: SonarSource → add "sonarsource" to IT services
# Fix 2: VAT Vakuumventile → add "vakuumventile" to Manufacturing  
# Fix 3: FIS Switzerland → add "zweigniederlassung" or "fis (switz" to Finance

# Update Manufacturing
BFS_INDUSTRIES[2] = ("10-33 Manufacturing", ["stadler", "siemens", "hitachi", "schindler",
                                               "swatch", "autoform", "bühler", "buhler",
                                               "jabil", "helbling technik", "bobst",
                                               "ruag", "agie charmilles", "mettler-toledo",
                                               "georg fischer", "alstom", "wolfensberger",
                                               "rollomatic", "starrag", "syngenta", "on ag",
                                               "liebherr", "sika ag", "wingtra",
                                               "winterthur gas", "medmix", "aeschlimann",
                                               "schleuniger", "sensirion", "viasat antenna",
                                               "sonceboz", "gf machining", "büchi labor",
                                               "büchi labortechnik", "novotec", "beyond gravity",
                                               "varo refining", "tecan", "zegna", "uhrteil",
                                               "vakuumventile", "swissto12", "pharmatronic",
                                               "huntsman", "maag pump", "fisba",
                                               "vf international", "beldona", "michael kors",
                                               "vilebrequin", "apco technologies", "h55"])

# Update Finance - add fis switzerland
BFS_INDUSTRIES[4] = ("64-66 Finance and Insurance", ["bank", "banque", "banking", "insurance",
                                               "versicherung", "asset management", "wealth",
                                               "investment fund", "pictet", "julius",
                                               "ubs", "efg", "comparis", "mobiliar",
                                               "axa", "helvetia", "baloise", "vontobel",
                                               "kantonalbank", "vaudoise", "groupe mutuel",
                                               "css", "swissquote", "avaloq", "lombard odier",
                                               "scor", "ficoba", "ppcmetrics", "wir bank",
                                               "bank cler", "bank frick", "prismalife",
                                               "orion rechtsschutz", "lgt bank", "finnova",
                                               "swissquant", "kepler cheuvreux", "swiss life",
                                               "vitol", "six group", "zweigniederlassung",
                                               "qim info", "stellenwerk", "anansi solutions"])

# Update IT services - add sonarsource
BFS_INDUSTRIES[9] = ("62-63 IT services", ["sonarsource", "sonar", "nexthink", "scandit",
                                             "salesforce", "servicenow", "infomaniak", "kudelski",
                                             "open systems", "infoguard", "ti&m", "frontify",
                                             "qoqa", "ch media", "bison schweiz",
                                             "speedcom", "exeon", "dexion", "extendis",
                                             "nexplore", "at rete", "cross-ing", "sigmalis",
                                             "inera sa", "systéa", "amag group", "enableme",
                                             "foodtechscout", "sapco", "philippe lebert",
                                             "manor ag", "daedalean", "supercomputing systems"])

# Re-run and verify
merged["industry"] = merged.apply(
    lambda row: classify_industry(row["company"], row["description"]), axis=1
)

for company in ["SonarSource SA", "VAT Vakuumventile AG",
                "FIS (Switzerland) SA, Zweigniederlassung Zürich"]:
    rows = merged[merged["company"] == company]
    if len(rows) > 0:
        print(f"{company} → {rows['industry'].iloc[0]}")

print("\nIndustry distribution:")
print(merged["industry"].value_counts())

SonarSource SA → 62-63 IT services
VAT Vakuumventile AG → 10-33 Manufacturing
FIS (Switzerland) SA, Zweigniederlassung Zürich → 64-66 Finance and Insurance

Industry distribution:
industry
62-63 IT services                            246
10-33 Manufacturing                           96
64-66 Finance and Insurance                   95
85 Education                                  77
86-88 Health and social work                  72
68-75 Real estate & professional services     52
49-53 Transport and storage                   36
84 Public administration                      31
41-43 Construction                            30
58-63 ICT                                      8
Name: count, dtype: int64


In [56]:
# Final merge with macro_industry
merged_final = merged.merge(
    macro_industry[["industry", "quarter", "vacancies"]].rename(columns={
        "quarter": "merge_quarter",
        "vacancies": "vacancies_industry"
    }),
    on=["industry", "merge_quarter"],
    how="left"
)

print("Shape before:", merged.shape)
print("Shape after:", merged_final.shape)
print("\nMatched rows (have industry vacancy data):", merged_final["vacancies_industry"].notna().sum())
print("Unmatched rows:", merged_final["vacancies_industry"].isna().sum())

# Save
merged_final.to_csv("../data/processed/jobs_merged_final.csv", index=False)
print("\nSaved! Final columns:", list(merged_final.columns))

Shape before: (743, 23)
Shape after: (743, 24)

Matched rows (have industry vacancy data): 740
Unmatched rows: 3

Saved! Final columns: ['job_id', 'title', 'company', 'role', 'location_raw', 'city', 'canton', 'region', 'posted_date', 'quarter', 'macro_quarter', 'seniority', 'workload_min', 'workload_max', 'salary_available', 'salary_min', 'salary_max', 'skills', 'skill_count', 'description', 'merge_quarter', 'vacancies_region', 'industry', 'vacancies_industry']


## Last Cleaning of Final Dataset 

1. only keep columns necessary for analysis
2. refine column names for better understanding
3. sort to ensure good flow of information

In [59]:
import pandas as pd

# Load
df = pd.read_csv("../data/processed/jobs_merged_final.csv")

# Select, reorder and rename columns
df = df[[
    # Job identifiers
    'job_id', 'title', 'company', 'role', 'seniority',
    # Location
    'city', 'canton', 'region',
    # Time
    'posted_date', 'quarter', 'merge_quarter',
    # Work conditions
    'workload_min', 'workload_max',
    'salary_available', 'salary_min', 'salary_max',
    # Skills
    'skills', 'skill_count',
    # Macro context
    'industry', 'vacancies_region', 'vacancies_industry',
    # Description
    'description',
]].rename(columns={
    'merge_quarter':      'macro_quarter',
    'vacancies_region':   'macro_vacancies_region',
    'vacancies_industry': 'macro_vacancies_industry',
})

# Check
print('Shape:', df.shape)
print('Columns:', list(df.columns))
print('Duplicates:', df.duplicated().sum())

Shape: (743, 22)
Columns: ['job_id', 'title', 'company', 'role', 'seniority', 'city', 'canton', 'region', 'posted_date', 'quarter', 'macro_quarter', 'workload_min', 'workload_max', 'salary_available', 'salary_min', 'salary_max', 'skills', 'skill_count', 'industry', 'macro_vacancies_region', 'macro_vacancies_industry', 'description']
Duplicates: 0


In [60]:
df.to_csv("../data/processed/jobs_merged_final.csv", index=False)
print("Saved!")

Saved!
